# Hex + AlphaGo-Style Self-Play: A Step-by-Step Walkthrough

This notebook walks through the full pipeline — game rules, network, search, self-play, training, and play — one piece at a time.

All the actual implementation (classes and functions) lives in **`utils.py`**, imported below. This notebook only calls into it; it does not redefine any logic. For the full narrative explanation behind every step (with worked numeric examples, diagrams, and design rationale), see **`HEX_IMPLEMENTATION.md`** in this repo — this notebook is the runnable companion to that document, section for section.

For background on *why* this architecture (policy + value network guiding MCTS) works at all, see `README.md`, which summarizes the original AlphaGo paper.

**Requirements:** `torch`, `numpy` (see `HEX_IMPLEMENTATION.md` Section 2).

All examples below use a tiny `board_size=3` or `5` and small simulation counts so every cell runs in seconds. Scale up `board_size`, `num_simulations`, and `num_iterations` for real training runs (see the Practical Tips table at the end of `HEX_IMPLEMENTATION.md`).

In [1]:
import numpy as np
import torch

from utils import (
    HexGame, RED, BLUE,
    HexNet,
    MCTS,
    self_play_game, generate_dataset,
    training_pipeline,
    human_vs_ai,
)

np.random.seed(0)
torch.manual_seed(0)
print("Ready. PyTorch:", torch.__version__)

Ready. PyTorch: 2.1.0+cu121


## 1. The Hex Game Environment

`HexGame` (in `utils.py`, `HEX_IMPLEMENTATION.md` Section 3) tracks the board, whose turn it is, and win detection via a Union-Find structure with 4 virtual edge nodes — a single `find(RED_TOP) == find(RED_BOTTOM)` check tells you if RED has won, in O(1).

Let's create a small board, look at it, and place a stone.

In [2]:
game = HexGame(size=3)
game.render()
print("\nLegal moves:", game.legal_moves())

game.make_move(0, 1)   # RED plays (0,1)
print("\nAfter RED plays (0,1):")
game.render()

. . .
 . . .
  . . .

Legal moves: [(0, 0), (0, 1), (0, 2), (1, 0), (1, 1), (1, 2), (2, 0), (2, 1), (2, 2)]

After RED plays (0,1):
. R .
 . . .
  . . .


### Win detection

Finishing the exact 5-move game traced in `HEX_IMPLEMENTATION.md` Section 3.3: RED plays `(0,1) -> (1,1) -> (2,0)`, BLUE plays `(0,0) -> (1,0)`. RED connects top row to bottom row through the center and wins.

In [3]:
for r, c in [(0, 0), (1, 1), (1, 0), (2, 0)]:
    game.make_move(r, c)

game.render()
print("\nWinner:", "RED" if game.winner == RED else "BLUE" if game.winner == BLUE else None)

B R .
 B R .
  R . .

Winner: RED


## 2. The Neural Network

`HexNet` (`HEX_IMPLEMENTATION.md` Section 4) is a residual trunk shared by two heads:

- **Input:** the current board state, encoded as `(2, N, N)` — channel 0 is always "whoever's turn it is," channel 1 is always "the opponent," regardless of color (Section 4.1).
- **Output:** a **policy** (`N²` raw logits, one per cell — which move looks best right now) and a **value** (one scalar in `[-1, +1]` — who's likely to win from here).

Let's feed a fresh board through an untrained network and look at both outputs.

In [5]:
board_size = 3
fresh_game = HexGame(size=board_size)

net = HexNet(board_size=board_size, num_channels=32, num_res_blocks=2)

state = torch.tensor(fresh_game.encode(), dtype=torch.float32).unsqueeze(0)  # (1, 2, N, N)
policy_logits, value = net(state)

print("Input shape:        ", state.shape)
print("Policy logits shape:", policy_logits.shape)
print("Value:              ", value.item())

print("Input values:\n", state)
print("Policy logits:\n", policy_logits)

Input shape:         torch.Size([1, 2, 3, 3])
Policy logits shape: torch.Size([1, 9])
Value:               -0.005751851946115494
Input values:
 tensor([[[[0., 0., 0.],
          [0., 0., 0.],
          [0., 0., 0.]],

         [[0., 0., 0.],
          [0., 0., 0.],
          [0., 0., 0.]]]])
Policy logits:
 tensor([[-1.8148e-01, -2.0202e-02,  4.0185e-02, -4.6100e-02,  6.6785e-02,
          3.0452e-02, -5.5223e-02, -4.4614e-05,  4.6359e-03]],
       grad_fn=<AddmmBackward0>)


## 3. Monte Carlo Tree Search

`MCTS` (`HEX_IMPLEMENTATION.md` Section 5) uses the network's policy as priors to decide which moves to explore, and its value to evaluate new leaf positions — instead of playing every simulation out to the end of the game. After many simulations, the **visit counts** at the root (not the raw policy) become the move distribution that self-play actually samples from.

Run a small search from the fresh board and look at the resulting visit distribution.

In [6]:
mcts = MCTS(net, num_simulations=100)
visits = mcts.search(fresh_game)

np.set_printoptions(precision=3, suppress=True)
print("Visit-count distribution (rows sum to 1):")
print(visits)
print("\nMost-visited cell:", np.unravel_index(visits.argmax(), visits.shape))

Visit-count distribution (rows sum to 1):
[[0.08 0.1  0.07]
 [0.09 0.28 0.09]
 [0.14 0.08 0.07]]

Most-visited cell: (1, 1)


## 4. Self-Play Data Generation

`self_play_game` (`HEX_IMPLEMENTATION.md` Section 6) plays one full game against itself, running MCTS at every move and sampling from the (temperature-adjusted) visit distribution rather than always taking the best move — so different games explore different lines. Each move produces one training record: `(state, mcts_policy, outcome)`, where `outcome` is filled in only after the game ends (+1 for the eventual winner's moves, -1 for the loser's).

`generate_dataset` just calls this many times and concatenates the results.

In [7]:
examples = self_play_game(net, board_size=3, num_simulations=25, temperature=1.0)

print(f"Game length: {len(examples)} moves\n")

state0, policy0, outcome0 = examples[0]
print("First record:")
print("  state shape: ", state0.shape)
print("  policy shape:", policy0.shape, "(sums to", round(policy0.sum(), 3), ")")
print("  outcome:     ", outcome0)

Game length: 9 moves

First record:
  state shape:  (2, 3, 3)
  policy shape: (3, 3) (sums to 1.0 )
  outcome:      1.0


In [8]:
dataset = generate_dataset(net, num_games=3, board_size=3, num_simulations=15)
print(f"\nTotal training examples from 3 self-play games: {len(dataset)}")


Total training examples from 3 self-play games: 25


## 5. Training

`training_pipeline` (`HEX_IMPLEMENTATION.md` Section 7) is the outer loop: generate self-play data with the *current* network, train on it for a few epochs, save a checkpoint, and repeat — each iteration's self-play uses whatever the network just learned (see Section 7.1's diagram).

The loss has two terms: cross-entropy between the network's policy and the MCTS visit distribution (`π_mcts`), plus MSE between the network's value and the actual game outcome (`z`). Section 7.2 walks one training step through with real numbers if you want the full mechanics.

Below is a *tiny* run — small board, few games, few simulations, one iteration — just to see the loop execute end to end in seconds. Scale every parameter up for a real training run.

In [ ]:
trained_net = training_pipeline(
    board_size        = 3,
    num_channels      = 16,
    num_res_blocks    = 1,
    num_iterations    = 2,
    games_per_iter    = 4,
    num_simulations   = 20,
    epochs_per_iter   = 2,
    batch_size        = 16,
    checkpoint_prefix = "hex_net_demo_iter",
)

## 6. Playing Against the Trained Network

`human_vs_ai` (`HEX_IMPLEMENTATION.md` Section 8) is interactive — it reads your moves from the keyboard — so it's best run directly in a notebook cell where you can type into the prompt (it won't do anything useful if you just "Run All" unattended). Uncomment the line below and run the cell to play against the network trained above.

In [ ]:
# human_vs_ai("hex_net_demo_iter002.pt", board_size=3, num_simulations=50)

## Next Steps

- Scale up: `board_size=11`, more channels/res blocks, more simulations, more iterations — see the **Practical Tips** table at the end of `HEX_IMPLEMENTATION.md`.
- Trained checkpoints (`*.pt`) and any exported datasets are excluded from git via `.gitignore` — copy the ones you want to keep somewhere outside this repo, or set up a proper model registry / release artifact if this grows into a longer-running project.
- Every function called above lives in `utils.py` — read it alongside `HEX_IMPLEMENTATION.md` if you want the line-by-line reasoning behind any piece.